# Speculative RAG — Draft First, Verify Second
## Draft Cheap, Verify Deep — The Speculative Retrieval Approach
⏱ ~15 min

**Speculative RAG** turns the standard retrieve-then-generate pipeline on its head. Instead of fetching documents before every answer, the model *drafts an answer first* using its training knowledge, then retrieves evidence only to verify (or correct) individual claims. The result: retrieval becomes a targeted fact-checker that fires only when the model might be wrong.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why Speculative RAG exists and when it wins |
| 2 | **Setup** — Vector store, LLMs, and the graph state |
| 3 | **Node 1: Draft** — Generate an answer from training knowledge |
| 4 | **Node 2: Extract Claims** — Parse a prose answer into checkable facts |
| 5 | **Node 3: Retrieve & Grade** — Fetch evidence and label each claim |
| 6 | **Node 4: Revise (conditional)** — Rewrite only the unsupported parts |
| 7 | **Full Pipeline** — Wire all four nodes into a LangGraph `StateGraph` |
| 8 | **Cost Analysis** — Measure retrieval savings vs always-retrieve RAG |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing the project dependencies, **or** Google Colab (install cell below handles it)
- `OPENAI_API_KEY` set in `.env` or Colab Secrets

### Key References
> Leviathan, Y., Kalman, M., & Matias, Y. (2023). *Fast Inference from Transformers via Speculative Decoding.* ICML 2023. https://arxiv.org/abs/2211.17192
>
> Rao, Z., et al. (2024). *Speculative RAG: Enhancing Retrieval Augmented Generation through Drafting.* https://arxiv.org/abs/2407.08223
>
> Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
>
> LangGraph documentation: https://langchain-ai.github.io/langgraph/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langchain-community",
            "langgraph",
            "chromadb",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or invalid.\n"
    "Local: add OPENAI_API_KEY=sk-... to .env\n"
    "Colab:  Secrets panel → Add secret 'OPENAI_API_KEY'"
)
print(f"OPENAI_API_KEY present: {bool(key)}  (ends ...{key[-4:]})")

---

## Part 1 — Concepts: Why Speculative RAG?

---

### The Cost Problem with Standard RAG

Standard RAG always retrieves — even for questions the LLM can answer perfectly from its training data. If a user asks *"What year did World War II end?"*, a standard RAG pipeline still:
1. Embeds the query (embedding API call)
2. Queries the vector store (disk/network I/O)
3. Fetches top-k documents (token cost in prompt)
4. Generates a grounded answer

Steps 1–3 are wasted when the model already knows the answer with high confidence.

---

### Speculative Decoding — The Inspiration

The technique borrows its name from **speculative decoding** (Leviathan et al., 2023), a hardware-efficiency trick: a small *drafter* model generates tokens speculatively; a large *verifier* model accepts or rejects them in parallel. If the drafter is right (often), you get large-model quality at small-model speed.

**Speculative RAG** (Rao et al., 2024) applies the same intuition to the retrieval problem:
- The LLM's **training knowledge** acts as the drafter — fast, cheap, often right
- **Retrieval + grading** acts as the verifier — targeted, only fires when needed

---

### Three Patterns Compared

| Pattern | Retrieval | Speed | Accuracy | Best for |
|---------|-----------|-------|----------|----------|
| **Standard RAG** | Always, before answer | Medium | High (if docs are good) | Unknown queries, private data |
| **Corrective RAG** | Always, then self-correct | Slow | Very high | High-stakes factual tasks |
| **Speculative RAG** | Only for unsupported claims | Fast | High | Mixed queries (some well-known, some niche) |
| **No RAG (LLM only)** | Never | Fastest | Varies — may hallucinate | Well-known, stable facts only |

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Draft** | The LLM's first answer generated from training knowledge, without any retrieval |
| **Claim** | One checkable factual statement extracted from the draft (e.g., "LangGraph was released in 2024") |
| **SUPPORTED** | Retrieval confirms the claim is consistent with the retrieved documents |
| **CONTRADICTED** | Retrieval shows the claim conflicts with the retrieved documents |
| **UNRELATED** | Retrieved documents do not address the claim — verdict is uncertain |
| **Revision** | The step that rewrites only the CONTRADICTED/UNRELATED claims using retrieved evidence |
| **Acceptance rate** | Fraction of claims that are SUPPORTED — analogous to token acceptance rate in speculative decoding |

### Speculative RAG Architecture

```
USER QUERY
    |
    v
+--------------------------------------+
|  Node 1: DRAFT                       |
|  LLM answers from training knowledge |  <- no retrieval here
|  "LangGraph was released in Jan 2024" |
+------------------+-------------------+
                   |
                   v
+--------------------------------------+
|  Node 2: EXTRACT CLAIMS              |
|  Parse prose into checkable facts    |
|  1. "Released January 2024"          |
|  2. "Built by LangChain Inc."        |
|  3. "Uses StateGraph API"            |
+------------------+-------------------+
                   |  (one retrieval call per claim)
                   v
+--------------------------------------+
|  Node 3: RETRIEVE & GRADE            |
|  For each claim:                     |
|   -> retrieve top-k docs from store  |
|   -> LLM grades: SUPPORTED /         |
|                  CONTRADICTED /      |
|                  UNRELATED           |
+--------+--------------+--------------+
         |              |
  all SUPPORTED    any unsupported
         |              |
         v              v
        END    +--------------------+
               |  Node 4: REVISE   |
               |  Rewrite only the |
               |  flagged claims   |
               |  using evidence   |
               +--------+----------+
                        |
                        v
                  FINAL ANSWER
```

> **Key insight:** When the LLM's training knowledge is correct, the graph reaches `END` after Node 3 — skipping all retrieval-in-the-prompt overhead. Retrieval only appears in the final answer when it is actually needed.

---

## Part 2 — Setup: Models, Vector Store, and State

---

### Design Decisions

| Component | Choice | Why |
|-----------|--------|-----|
| LLM (all nodes) | `gpt-4o-mini` | Fast, cheap, strong enough for claim extraction + grading |
| Embeddings | `text-embedding-3-small` | Low cost, 1536 dims, excellent for semantic retrieval |
| Vector store | ChromaDB (in-memory) | No server needed — perfect for notebooks |
| Graph framework | LangGraph `StateGraph` | Handles conditional routing (`revise` vs `END`) cleanly |
| Temperature | 0 | Deterministic grading — SUPPORTED/CONTRADICTED/UNRELATED must be consistent |

In [ ]:
# ─── 2-A: Core imports ────────────────────────────────────────────────────────
# TypedDict gives LangGraph typed access to state keys with zero runtime overhead — no class instantiation.
from typing import TypedDict

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
# StrOutputParser strips the AIMessage wrapper so nodes receive a plain string — avoids .content access everywhere.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import END, StateGraph

print("Imports OK")

In [ ]:
# ─── 2-B: Models and embeddings ───────────────────────────────────────────────
# temperature=0 is critical for the grading node — we need deterministic labels.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"LLM:        {llm.model_name}")
print(f"Embeddings: {embeddings.model}")

In [ ]:
# ─── 2-C: Build the knowledge base ───────────────────────────────────────────
# A small corpus about LangGraph and LangChain.
# We will use this throughout the workshop. Feel free to swap in your own docs.

KNOWLEDGE_DOCS = [
    "LangGraph was created by LangChain Inc. and released in January 2024.",
    "LangGraph is built on top of LangChain and uses its message and tool abstractions.",
    "LangGraph supports human-in-the-loop workflows via interrupt() and Command(resume=...).",
    "LangGraph StateGraph compiles into a runnable that can be invoked or streamed.",
    "LangChain was founded by Harrison Chase in 2022 and raised a Series A in 2023.",
    "MemorySaver is a built-in LangGraph checkpointer that stores state in memory.",
    "LangGraph supports parallel execution using the Send API for map-reduce patterns.",
    "The LangGraph prebuilt create_react_agent handles the ReAct loop automatically.",
    "LangChain's LCEL (LangChain Expression Language) uses the | pipe operator for chains.",
    "LangGraph graphs can have conditional edges that route based on node return values.",
]

docs = [Document(page_content=d) for d in KNOWLEDGE_DOCS]
vectorstore = Chroma.from_documents(docs, embeddings)
# k=2: low recall keeps per-claim cost down; raise to 4-5 if your corpus is large and claims are broad.
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"Vector store loaded: {len(KNOWLEDGE_DOCS)} documents")
print("Retriever configured: k=2 (top-2 docs per claim)")

In [ ]:
# ─── 2-D: Define the graph state ─────────────────────────────────────────────
# LangGraph passes this TypedDict between nodes.
# Each node returns a dict of the keys it updates — others are left unchanged.

# Each node returns only the keys it mutates; LangGraph merges them into the shared state dict.
class SpeculativeState(TypedDict):
    query: str            # The user's question
    draft: str            # LLM's first answer (no retrieval)
    claims: list[str]     # Individual checkable facts extracted from the draft
    evidence: list[str]   # Retrieved doc text for each claim
    support_labels: list[str]  # SUPPORTED / CONTRADICTED / UNRELATED per claim
    revised: str          # Final answer (empty string if no revision needed)


# Inspect the fields
print("SpeculativeState fields:")
for field, annotation in SpeculativeState.__annotations__.items():
    print(f"  {field:20} -> {annotation}")

---

## Part 3 — Node 1: Draft

---

The `draft` node is the heart of what makes this pattern speculative. Instead of retrieving first, we ask the LLM to answer the question directly from its training knowledge.

### Why "speculative"?

The draft is a *speculation* — the model's best guess based on what it learned during training. It might be:
- **Correct** — retrieval will confirm it, and we skip the revise step entirely
- **Partially wrong** — retrieval will catch the wrong claims and revision will fix them
- **Completely wrong** — all claims get flagged, revision rewrites the whole answer

The critical design choice: the draft node makes **zero retrieval calls**. Its speed advantage comes entirely from avoiding the embed → search → fetch round-trip.

### System prompt design

The draft prompt should encourage **specific, factual claims** — vague hedging produces claims that are hard to verify. Ask for concrete details (dates, names, version numbers) because those are exactly what retrieval can check.

In [ ]:
# ─── 3-A: Draft node ─────────────────────────────────────────────────────────
# Generates an answer from training knowledge — no retrieval.

# "Do not hedge" is intentional — hedged answers produce vague claims that are harder to verify.
DRAFT_SYSTEM = (
    "You are a knowledgeable AI assistant. Answer the question in 3-4 sentences "
    "using your training knowledge. Be specific: include dates, names, and version "
    "numbers where you know them. Do not hedge — make concrete claims."
)


def draft(state: SpeculativeState) -> dict:
    """Generate an answer directly from LLM training knowledge — no retrieval."""
    result = llm.invoke([
        SystemMessage(DRAFT_SYSTEM),
        HumanMessage(state["query"]),
    ])
    print(f"  [draft] {result.content[:100]}...")
    return {
        "draft": result.content,
        "claims": [],
        "evidence": [],
        "support_labels": [],
        "revised": "",
    }


# Test the draft node in isolation
test_state: SpeculativeState = {
    "query": "When was LangGraph released and who made it?",
    "draft": "", "claims": [], "evidence": [], "support_labels": [], "revised": "",
}
draft_output = draft(test_state)
print()
print("Full draft:")
print(draft_output["draft"])

In [ ]:
# ─── 3-B: Compare draft vs standard RAG on the same question ─────────────────
# This cell shows why a good draft is valuable — and when it needs correction.

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


standard_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the context below.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

standard_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | standard_prompt
    | llm
    | StrOutputParser()
)

query = "When was LangGraph released and who made it?"

print("STANDARD RAG (retrieve first, then answer):")
std_answer = standard_chain.invoke(query)
print(std_answer)

print()
print("SPECULATIVE RAG DRAFT (answer first, no retrieval):")
print(draft_output["draft"])

print()
print("Observation: both answers may be very similar.")
print("The draft's advantage is speed — it skips the retrieval round-trip.")
print("Speculative RAG's value appears when the draft contains a wrong claim.")

---

## Part 4 — Node 2: Extract Claims

---

A prose answer like *"LangGraph was built by LangChain Inc. in early 2024 and supports stateful multi-agent pipelines"* contains three verifiable claims:

1. Built by LangChain Inc.
2. Released in early 2024
3. Supports stateful multi-agent pipelines

The `extract_claims` node uses the LLM to decompose the draft into this numbered list. Each claim is then independently retrieved against and labeled in the next node.

### Why parse into claims?

If we retrieved against the whole draft at once, we would get back a mix of relevant and irrelevant docs. Claim-level retrieval is **surgically precise** — each claim gets the most relevant evidence for its specific fact. This dramatically improves the accuracy of SUPPORTED/CONTRADICTED/UNRELATED labels.

### Prompt engineering tip

Instruct the LLM to extract claims as **"Output ONLY the numbered list"** — otherwise models tend to add preamble ("Here are the factual claims:") that makes parsing fragile.

In [ ]:
# ─── 4-A: Extract claims node ─────────────────────────────────────────────────

CLAIM_SYSTEM = (
    "Extract the distinct factual claims from the given answer as a numbered list. "
    "Each item should be a single, self-contained, checkable fact. "
    "Output ONLY the numbered list — no preamble, no explanation."
)


def extract_claims(state: SpeculativeState) -> dict:
    """Parse the draft into individual checkable facts."""
    result = llm.invoke([
        SystemMessage(CLAIM_SYSTEM),
        HumanMessage(f"Answer to fact-check:\n{state['draft']}"),
    ])
    lines = [l.strip() for l in result.content.strip().split("\n") if l.strip()]
# Fragile but intentional: numbered-list output is easy to parse without structured_output overhead.
    # Strip leading "1. " / "2. " prefixes
    claims = [l.split(". ", 1)[-1] if l[0].isdigit() else l for l in lines]
    print(f"  [extract_claims] {len(claims)} claims extracted")
    for i, c in enumerate(claims, 1):
        print(f"    {i}. {c}")
    return {"claims": claims}


# Test: pipe the draft output into extract_claims
state_with_draft = {**test_state, **draft_output}
claims_output = extract_claims(state_with_draft)
print()
print(f"Total claims: {len(claims_output['claims'])}")

In [ ]:
# ─── 4-B: Inspect claim granularity ─────────────────────────────────────────
# Too coarse: claims that bundle multiple facts (hard to verify cleanly)
# Too fine:   trivially checkable or unfalsifiable claims
# Ideal:      one specific, distinct fact per claim

example_drafts = [
    (
        "Dense answer",
        "LangGraph, released in January 2024 by LangChain Inc., is a framework "
        "for building stateful multi-agent workflows. It is built on LangChain "
        "and uses a StateGraph API where nodes are Python functions.",
    ),
    (
        "Vague answer",
        "LangGraph is a tool made by the LangChain team that helps people build AI applications.",
    ),
]

for label, draft_text in example_drafts:
    result = llm.invoke([SystemMessage(CLAIM_SYSTEM), HumanMessage(f"Answer:\n{draft_text}")])
    lines = [l.strip() for l in result.content.strip().split("\n") if l.strip()]
    claims = [l.split(". ", 1)[-1] if l[0].isdigit() else l for l in lines]
    print(f"{label}: {len(claims)} claims")
    for c in claims:
        print(f"  - {c}")
    print()

---

## Part 5 — Node 3: Retrieve & Grade

---

This is the most expensive node — it makes one retrieval call **per claim**. The key insight is that this is still often cheaper than standard RAG: if a query produces 4 claims and 3 of them are SUPPORTED, the final answer only needs retrieval evidence for 1 claim. And for queries where all claims are SUPPORTED, **zero retrieved documents appear in the LLM's context window** (they were only used for grading).

### The grading labels

```
Claim: "LangGraph was released in January 2024"
Context: "LangGraph was created by LangChain Inc. and released in January 2024."
-> SUPPORTED    context confirms the claim

Claim: "LangGraph was released in March 2023"
Context: "LangGraph was created by LangChain Inc. and released in January 2024."
-> CONTRADICTED  context explicitly conflicts with the claim

Claim: "LangGraph uses Redis for persistence"
Context: "MemorySaver is a built-in LangGraph checkpointer that stores state in memory."
-> UNRELATED    context does not address this claim directly
```

### Cost tradeoffs by acceptance rate

| Scenario | Retrieval calls | Docs in final prompt | Revision triggered |
|----------|----------------|---------------------|--------------------|
| All SUPPORTED | N (grading only) | 0 | No |
| Mixed (some wrong) | N | Partial | Yes |
| All CONTRADICTED | N | All N | Yes |

The savings are real: when the model is right, retrieved docs never enter the final generation prompt.

In [ ]:
# ─── 5-A: Retrieve and grade node ─────────────────────────────────────────────

SUPPORT_SYSTEM = (
    "You are a fact-checker. Given a claim and retrieved context, determine whether "
    "the context SUPPORTS, CONTRADICTS, or is UNRELATED to the claim. "
    "Reply with exactly one word: SUPPORTED, CONTRADICTED, or UNRELATED."
)


def retrieve_and_grade(state: SpeculativeState) -> dict:
    """Retrieve evidence for each claim and label it SUPPORTED/CONTRADICTED/UNRELATED."""
    evidence_list, label_list = [], []
# One retrieval call per claim — this is the expensive part; cost scales linearly with claim count.
    for i, claim in enumerate(state["claims"], 1):
        # Retrieve top-k docs most relevant to this specific claim
        docs = retriever.invoke(claim)
        context = "\n".join(d.page_content for d in docs)
        # Ask the LLM to grade the claim against the retrieved context
# Single-word reply (SUPPORTED/CONTRADICTED/UNRELATED) avoids JSON parsing and structured_output cost.
        label_response = llm.invoke([
            SystemMessage(SUPPORT_SYSTEM),
            HumanMessage(f"Claim: {claim}\n\nContext:\n{context}"),
        ])
        label = label_response.content.strip().upper()
        evidence_list.append(context)
        label_list.append(label)
        print(f"  [{label:14}] {claim[:70]}")
    return {"evidence": evidence_list, "support_labels": label_list}


# Test the grade node
state_with_claims = {**state_with_draft, **claims_output}
grade_output = retrieve_and_grade(state_with_claims)
print()
summary = {label: grade_output["support_labels"].count(label)
           for label in ["SUPPORTED", "CONTRADICTED", "UNRELATED"]}
print(f"Summary: {summary}")

In [ ]:
# ─── 5-B: Inspect the raw retrieval evidence ─────────────────────────────────
# This is the key debugging step: see exactly what evidence each claim retrieved.

print("=== CLAIM-BY-CLAIM EVIDENCE ===")
for i, (claim, label, evidence) in enumerate(
    zip(
        state_with_claims["claims"],
        grade_output["support_labels"],
        grade_output["evidence"],
    ),
    1,
):
    print(f"\nClaim {i} [{label}]: {claim}")
    print("Evidence retrieved:")
    for line in evidence.split("\n"):
        if line.strip():
            print(f"  > {line}")

---

## Part 6 — Node 4: Revise (Conditional)

---

The `revise` node fires **only when at least one claim is not SUPPORTED**. It receives:
- The original draft
- A full evidence block (claim + label + retrieved text for each claim)

Its job is surgical: **keep everything that is SUPPORTED, rewrite only what is CONTRADICTED or UNRELATED**.

This is where the quality guarantee comes from. Even if the draft was mostly correct, the revision node can pinpoint the one wrong sentence and fix it using evidence — rather than regenerating the entire answer from scratch.

### The routing logic

```python
def needs_revision(state) -> str:
    # LangGraph conditional edge: return the name of the next node
    if any(l != "SUPPORTED" for l in state["support_labels"]):
        return "revise"   # -> Node 4
    return "end"          # -> END (skip revision entirely)
```

The string returned maps to LangGraph's conditional edge dict:
```python
graph.add_conditional_edges(
    "retrieve_and_grade",
    needs_revision,
    {"revise": "revise", "end": END}
)
```

In [ ]:
# ─── 6-A: Revise node and routing function ────────────────────────────────────

REVISE_SYSTEM = (
    "You are a precise editor. Given an original answer and a fact-check report, "
    "revise ONLY the parts that are CONTRADICTED or UNRELATED to the evidence. "
    "Keep SUPPORTED content word-for-word unchanged. "
    "Return the complete revised answer."
)


# add_conditional_edges maps the return string of this function to the next node name — "end" maps to END.
def needs_revision(state: SpeculativeState) -> str:
    """Conditional edge function: 'revise' if any claim failed, 'end' if all passed."""
    unsupported = [l for l in state["support_labels"] if l != "SUPPORTED"]
    verdict = "revise" if unsupported else "end"
    print(f"  [router] labels={state['support_labels']} -> {verdict}")
    return verdict


def revise(state: SpeculativeState) -> dict:
    """Rewrite only the unsupported parts of the draft using retrieved evidence."""
    evidence_block = "\n\n".join(
        f"Claim: {c}\nStatus: {l}\nEvidence: {e}"
        for c, l, e in zip(state["claims"], state["support_labels"], state["evidence"])
    )
    result = llm.invoke([
        SystemMessage(REVISE_SYSTEM),
        HumanMessage(
            f"Original answer:\n{state['draft']}\n\n"
            f"Fact-check report:\n{evidence_block}"
        ),
    ])
    print("  [revise] answer updated based on evidence")
    return {"revised": result.content}


# Test revise in isolation with a deliberately wrong claim
state_for_revise_test = {
    **state_with_claims,
    **grade_output,
    "claims": ["LangGraph was released in March 2023"],
    "evidence": ["LangGraph was created by LangChain Inc. and released in January 2024."],
    "support_labels": ["CONTRADICTED"],
    "draft": "LangGraph was released in March 2023 by LangChain Inc.",
}

print("Testing revise with a CONTRADICTED claim:")
revise_output = revise(state_for_revise_test)
print("Revised answer:")
print(revise_output["revised"])

In [ ]:
# ─── 6-B: Demonstrate skip-revision path ─────────────────────────────────────
# When all claims are SUPPORTED the graph goes directly to END.
# The revision node never runs — saving one LLM call.

all_supported_state = {
    **state_with_claims,
    "support_labels": ["SUPPORTED", "SUPPORTED", "SUPPORTED"],
    "evidence": ["", "", ""],
}

print("Routing with all-SUPPORTED labels:")
route = needs_revision(all_supported_state)
print(f"Next node: {route}")
print("The revise() function never runs. Draft becomes the final answer.")
print()

mixed_state = {
    **state_with_claims,
    "support_labels": ["SUPPORTED", "CONTRADICTED", "SUPPORTED"],
    "evidence": ["", "", ""],
}

print("Routing with one CONTRADICTED label:")
route = needs_revision(mixed_state)
print(f"Next node: {route}")
print("The revise() function runs and rewrites only the contradicted claim.")

---

## Part 7 — Full Pipeline: Compile and Run the Graph

---

Now we wire all four nodes into a LangGraph `StateGraph`. The compiled graph is a single callable that handles all routing automatically.

### Graph topology

```
START
  |
  v
draft --> extract_claims --> retrieve_and_grade
                                    |
                       +------------+------------------+
                  all SUPPORTED              any unsupported
                       |                          |
                       v                          v
                      END                      revise --> END
```

The `add_conditional_edges` call is what makes the graph branch at `retrieve_and_grade`. The function `needs_revision` returns a string key that maps to the next node name via the routing dict.

In [ ]:
# ─── 7-A: Compile the full LangGraph graph ────────────────────────────────────

graph = StateGraph(SpeculativeState)

# Register nodes
graph.add_node("draft", draft)
graph.add_node("extract_claims", extract_claims)
graph.add_node("retrieve_and_grade", retrieve_and_grade)
graph.add_node("revise", revise)

# Wire edges
graph.set_entry_point("draft")
graph.add_edge("draft", "extract_claims")
graph.add_edge("extract_claims", "retrieve_and_grade")
# The dict maps needs_revision() return values to node names — makes routing explicit and inspectable.
graph.add_conditional_edges(
    "retrieve_and_grade",
    needs_revision,
    {"revise": "revise", "end": END},
)
graph.add_edge("revise", END)

# compile() locks the topology and returns a runnable Pregel executor — the graph cannot be modified after this.
app = graph.compile()
print("Graph compiled successfully.")
print()
print("Node execution order:")
print("  draft -> extract_claims -> retrieve_and_grade -> (revise | END)")

In [ ]:
# ─── 7-B: Run the full pipeline ──────────────────────────────────────────────

def run_speculative_rag(query: str) -> dict:
    """Run the speculative RAG pipeline and return the full state."""
    initial_state: SpeculativeState = {
        "query": query,
        "draft": "",
        "claims": [],
        "evidence": [],
        "support_labels": [],
        "revised": "",
    }
    return app.invoke(initial_state)


QUERY = "When was LangGraph released and who made it?"

print(f"QUERY: {QUERY}")
print("=" * 60)
result = run_speculative_rag(QUERY)

print()
print("DRAFT:")
print(result["draft"])

print()
print("CLAIMS & VERDICTS:")
for claim, label in zip(result["claims"], result["support_labels"]):
    icon = "[OK] " if label == "SUPPORTED" else ("[X]  " if label == "CONTRADICTED" else "[?]  ")
    print(f"  {icon}[{label}] {claim}")

print()
if result["revised"]:
    print("REVISED ANSWER (some claims were corrected):")
    print(result["revised"])
else:
    print("FINAL ANSWER = DRAFT (all claims supported — no revision needed)")
    print(result["draft"])

In [ ]:
# ─── 7-C: Run multiple queries and observe routing differences ────────────────

SAMPLE_QUERIES = [
    "When was LangGraph released and who made it?",
    "How does LangGraph handle human-in-the-loop workflows?",
    "What is the capital of France?",  # Not in knowledge base -> expect UNRELATED
]

for query in SAMPLE_QUERIES:
    print(f"\nQUERY: {query}")
    print("─" * 60)
    r = run_speculative_rag(query)
    n_supported = r["support_labels"].count("SUPPORTED")
    n_total = len(r["support_labels"])
    acceptance_rate = n_supported / n_total if n_total > 0 else 0
    revised = bool(r["revised"])
    print(f"  Claims: {n_total}  |  Supported: {n_supported}  |  Acceptance rate: {acceptance_rate:.0%}")
    print(f"  Revision triggered: {revised}")
    final = r["revised"] if revised else r["draft"]
    print(f"  Final answer: {final[:120]}...")

---

## Part 8 — Cost Analysis: When Does Speculative RAG Save Work?

---

Speculative RAG's efficiency depends on the **claim acceptance rate** — what fraction of the draft's claims are confirmed SUPPORTED by retrieval.

### Acceptance rate and cost

| Acceptance rate | What happens | vs Standard RAG |
|----------------|-------------|------------------|
| 100% (all SUPPORTED) | No revision — draft is final answer | Saves: revision LLM call + context tokens |
| ~80% (most SUPPORTED) | Revision touches a few sentences | Similar cost |
| ~50% (mixed) | Significant revision needed | Roughly equal |
| ~0% (all UNRELATED) | Full rewrite from evidence | More expensive — two generation steps |

### When to use each pattern

```
Query type                              Recommended pattern
────────────────────────────────────────────────────────────────
Well-known facts (model confident)  ->  Speculative RAG (high acceptance rate)
Private/proprietary data            ->  Standard RAG (model has no training signal)
High-stakes, zero hallucination     ->  Corrective RAG or Self-RAG
Mixed workload                      ->  Speculative RAG (best average-case cost)
```

In [ ]:
# ─── 8-A: Measure acceptance rates across a query batch ───────────────────────

benchmark_queries = [
    "When was LangGraph released and who made it?",
    "How does LangGraph handle human-in-the-loop workflows?",
    "What is MemorySaver in LangGraph?",
    "How does LangGraph support parallel execution?",
    "What is LCEL and how does it work?",
]

total_claims = 0
total_supported = 0
revisions_needed = 0

print(f"{'Query':50} {'Claims':7} {'OK':5} {'Rate':7} {'Revised?'}")
print("-" * 85)

for q in benchmark_queries:
    r = run_speculative_rag(q)
    n = len(r["support_labels"])
    s = r["support_labels"].count("SUPPORTED")
    rev = bool(r["revised"])
    rate = s / n if n > 0 else 0
    total_claims += n
    total_supported += s
    if rev:
        revisions_needed += 1
    print(f"{q[:49]:50} {n:7} {s:5} {rate:7.0%} {'YES' if rev else 'no'}")

overall_rate = total_supported / total_claims if total_claims > 0 else 0
print("-" * 85)
print(f"{'OVERALL':50} {total_claims:7} {total_supported:5} {overall_rate:7.0%} {revisions_needed}/{len(benchmark_queries)} revised")
print()
print(f"Retrieval calls made:   {total_claims} (one per claim)")
print(f"Revision LLM calls:     {revisions_needed}")
print(f"Revision calls saved:   {len(benchmark_queries) - revisions_needed} (vs always-revise)")

---

## Exercises

---

### Exercise 1 — Inject a wrong fact

Add `"LangGraph was founded in 2019."` to `KNOWLEDGE_DOCS` and rebuild the vector store. Then ask *"When was LangGraph founded?"*. Does `CONTRADICTED` appear? Does the revision fix the date correctly?

**Goal:** Confirm that the CONTRADICTED path actually corrects wrong claims.

### Exercise 2 — Ask something outside the knowledge base

Ask *"What is the capital of Australia?"* — a question the model knows from training but the knowledge base does not cover. Observe:
- Are claims labeled UNRELATED (no relevant docs) or SUPPORTED (coincidental match)?
- Does revision trigger? What does the revised answer look like?

### Exercise 3 — Count retrieval calls

Add a counter to `retrieve_and_grade` to track total retrieval calls across 5 queries. Compare to a standard RAG pipeline that retrieves once per query with k=3. Which is cheaper in retrieval calls?

### Exercise 4 — Your own corpus

Replace `KNOWLEDGE_DOCS` with 10 sentences from any Wikipedia article on a topic the LLM knows well (e.g., Python programming language, space exploration). Run 3 queries:
1. A question the model almost certainly knows correctly
2. A specific fact buried in your 10 sentences
3. A question your 10 sentences cannot answer

Observe the acceptance rate pattern.

### Exercise 5 (Advanced) — Vary k

Rebuild the retriever with `k=1`, `k=3`, and `k=5`. Does a higher k improve SUPPORTED rates, or does it introduce noise that causes correct claims to be labeled CONTRADICTED?

In [ ]:
# ── Exercise 1 Starter — inject a wrong fact ──────────────────────────────────
# Add a deliberately incorrect document and rebuild the vector store.

WRONG_DOCS = KNOWLEDGE_DOCS + [
    "LangGraph was founded in 2019 as an open-source project.",  # wrong year
]

# TODO: rebuild vectorstore_ex1 from WRONG_DOCS
# vectorstore_ex1 = Chroma.from_documents([Document(page_content=d) for d in WRONG_DOCS], embeddings)

# TODO: rebuild retriever_ex1 with k=2
# retriever_ex1 = vectorstore_ex1.as_retriever(search_kwargs={"k": 2})

# TODO: rebuild a retrieve_and_grade node that uses retriever_ex1
# TODO: compile a new graph app_ex1 with the updated node

# TODO: run the pipeline on the query below
# result_ex1 = app_ex1.invoke({"query": "When was LangGraph founded?", ...})

# EXPECTED: one claim labeled CONTRADICTED, revision corrects the year to January 2024
print("TODO: complete the steps above, then re-run.")

In [ ]:
# ── Exercise 3 Starter — count retrieval calls ────────────────────────────────

retrieval_call_count = 0


def retrieve_and_grade_counted(state: SpeculativeState) -> dict:
    global retrieval_call_count
    evidence_list, label_list = [], []
    for claim in state["claims"]:
        retrieval_call_count += 1  # track every retrieval
        docs = retriever.invoke(claim)
        context = "\n".join(d.page_content for d in docs)
        label_response = llm.invoke([
            SystemMessage(SUPPORT_SYSTEM),
            HumanMessage(f"Claim: {claim}\n\nContext:\n{context}"),
        ])
        evidence_list.append(context)
        label_list.append(label_response.content.strip().upper())
    return {"evidence": evidence_list, "support_labels": label_list}


# TODO: rebuild graph_counted using retrieve_and_grade_counted instead of retrieve_and_grade
# TODO: run 5 queries through graph_counted
# TODO: compare retrieval_call_count to (5 queries x k=3) = 15 for standard RAG

print("retrieval_call_count starts at:", retrieval_call_count)
print("TODO: wire retrieve_and_grade_counted into the graph and run 5 queries.")
print("Then compare: retrieval_call_count vs 5 x 3 = 15 (standard RAG with k=3)")

---

## Answer Key

---

Use this section after attempting the exercises. These are sample solutions — your exact output will vary based on LLM responses.

### Exercise 1 — Inject a wrong fact

**What to expect:** The `retrieve_and_grade` node retrieves both the correct document (`"released in January 2024"`) and the injected wrong document (`"founded in 2019"`) when grading a claim about the founding date. The LLM should label that claim CONTRADICTED because the two documents conflict. The `revise` node should then produce an answer that says "January 2024" rather than "2019".

**If you see UNRELATED instead of CONTRADICTED:** The wrong document may not have been retrieved for that specific claim. Try increasing k to 3 or rewriting the wrong document to more closely match the query wording.

### Exercise 2 — Outside-the-knowledge-base query

**What to expect:** Asking *"What is the capital of Australia?"* produces claims labeled UNRELATED (the LangGraph knowledge base has no Australia docs). The revision node fires because UNRELATED counts as unsupported, but the revised answer is based on weak evidence — the model may produce a hedged or hallucinated revision.

**Key lesson:** UNRELATED claims are the weakest case for Speculative RAG. If your knowledge base does not cover the topic, retrieval cannot verify or correct — and revision may make things worse. Knowledge base coverage matters as much as the graph architecture.

### Exercise 3 — Count retrieval calls

**What to expect:** With 5 queries averaging ~3 claims each, speculative RAG makes roughly 15 retrieval calls — the same as standard RAG with k=3. However, standard RAG also feeds all 15 retrieved documents into the generation prompt. Speculative RAG only feeds evidence for UNRELATED/CONTRADICTED claims into the revision prompt — so the context window savings are significant even when retrieval call counts are similar.

### Exercise 5 — Vary k

**What to expect:** k=1 is often too aggressive — a single mismatched doc can cause a correct claim to be labeled UNRELATED. k=2 is the sweet spot for a 10-doc corpus. k=3+ provides diminishing returns and increases context size in the grading prompt. For larger corpora (1000+ docs), k=3–5 is typically better.

In [ ]:
# Sample solution for Exercise 1
wrong_docs_list = [
    Document(page_content=d)
    for d in KNOWLEDGE_DOCS + ["LangGraph was founded in 2019 as an open-source project."]
]
vectorstore_ex1 = Chroma.from_documents(wrong_docs_list, embeddings)
retriever_ex1 = vectorstore_ex1.as_retriever(search_kwargs={"k": 3})


def retrieve_and_grade_ex1(state: SpeculativeState) -> dict:
    evidence_list, label_list = [], []
    for claim in state["claims"]:
        docs = retriever_ex1.invoke(claim)
        context = "\n".join(d.page_content for d in docs)
        label_response = llm.invoke([
            SystemMessage(SUPPORT_SYSTEM),
            HumanMessage(f"Claim: {claim}\n\nContext:\n{context}"),
        ])
        evidence_list.append(context)
        label_list.append(label_response.content.strip().upper())
    return {"evidence": evidence_list, "support_labels": label_list}


graph_ex1 = StateGraph(SpeculativeState)
graph_ex1.add_node("draft", draft)
graph_ex1.add_node("extract_claims", extract_claims)
graph_ex1.add_node("retrieve_and_grade", retrieve_and_grade_ex1)
graph_ex1.add_node("revise", revise)
graph_ex1.set_entry_point("draft")
graph_ex1.add_edge("draft", "extract_claims")
graph_ex1.add_edge("extract_claims", "retrieve_and_grade")
graph_ex1.add_conditional_edges("retrieve_and_grade", needs_revision, {"revise": "revise", "end": END})
graph_ex1.add_edge("revise", END)
app_ex1 = graph_ex1.compile()

result_ex1 = app_ex1.invoke({
    "query": "When was LangGraph founded?",
    "draft": "", "claims": [], "evidence": [], "support_labels": [], "revised": "",
})

print("Exercise 1 result:")
for claim, label in zip(result_ex1["claims"], result_ex1["support_labels"]):
    icon = "[OK]" if label == "SUPPORTED" else ("[X]" if label == "CONTRADICTED" else "[?]")
    print(f"  {icon} [{label}] {claim}")
if result_ex1["revised"]:
    print("\nRevised answer:")
    print(result_ex1["revised"])

---

## What's Next?

---

You now understand Speculative RAG: draft first → extract claims → retrieve per claim → conditional revision. Here is where to go deeper:

### Related patterns in this workshop series

- **Example 17 — Corrective RAG** (`../17-corrective-rag/`): retrieve first, then self-correct if the retrieved docs are poor quality — complements Speculative RAG for high-stakes domains
- **Example 27 — Self-RAG** (`../27-self-rag/`): decide per sentence whether to retrieve — a finer-grained version of the speculative approach
- **Example 25 — Adaptive RAG** (`../25-adaptive-rag/`): classify the query upfront to pick the cheapest retrieval strategy — Speculative RAG as one branch of a multi-strategy router
- **Example 12 — Basic RAG Notebook** (`../12-basic-rag-notebook/`): review the foundational retrieve-then-generate pipeline that Speculative RAG improves upon

### Production improvements

- **Parallel claim grading**: use `asyncio.gather()` or LangGraph's Send API to grade all claims in parallel — reduces latency from O(n_claims) sequential to O(1) round-trips
- **Confidence threshold**: instead of a binary SUPPORTED/NOT, ask the grader for a 0–1 confidence score and only revise claims below a threshold
- **Caching**: if the same claim appears in multiple queries (common for entity-heavy workloads), cache the SUPPORTED label and skip retrieval
- **Streaming**: LangGraph supports `.astream()` — yield the draft immediately for low-latency UX, then stream revisions as they arrive

### Academic references

> Leviathan, Y., Kalman, M., & Matias, Y. (2023). *Fast Inference from Transformers via Speculative Decoding.* ICML 2023. https://arxiv.org/abs/2211.17192
>
> Rao, Z., et al. (2024). *Speculative RAG: Enhancing Retrieval Augmented Generation through Drafting.* https://arxiv.org/abs/2407.08223
>
> Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
>
> Asai, A., et al. (2023). *Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection.* https://arxiv.org/abs/2310.11511
>
> Shi, F., et al. (2023). *REPLUG: Retrieval-Augmented Black-Box Language Models.* https://arxiv.org/abs/2301.12652

---

*Workshop complete. Try example 17 (Corrective RAG) next — it retrieves first and corrects after, complementing the speculative approach you just learned.*